# HW02 解答

## 2. 多层感知机 (MLP)

### 2.1 理论部分

#### 1. 线性模型级联证明
**问题**：证明两个线性模型的级联仍是一个线性模型。

**证明**：
给定：
$h = \mathbf{W}_1 \mathbf{x} + \mathbf{b}_1$
$o = \mathbf{W}_2 \mathbf{h} + \mathbf{b}_2$

将 $h$ 代入 $o$ 的表达式中：
$o = \mathbf{W}_2 (\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2$
$o = (\mathbf{W}_2 \mathbf{W}_1) \mathbf{x} + (\mathbf{W}_2 \mathbf{b}_1 + \mathbf{b}_2)$

令 $\mathbf{W}' = \mathbf{W}_2 \mathbf{W}_1$ 且 $\mathbf{b}' = \mathbf{W}_2 \mathbf{b}_1 + \mathbf{b}_2$。
则 $o = \mathbf{W}' \mathbf{x} + \mathbf{b}'$，这显然是一个线性模型的形式。

#### 2. 激活函数导数推导
- **Sigmoid 函数**: $\sigma(x) = \frac{1}{1 + e^{-x}}$
  导数：$\sigma'(x) = \frac{e^{-x}}{(1 + e^{-x})^2} = \frac{1}{1 + e^{-x}} \cdot \frac{e^{-x}}{1 + e^{-x}} = \sigma(x)(1 - \sigma(x))$

- **tanh 函数**: $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$
  导数：$\tanh'(x) = 1 - \tanh^2(x)$

### 2.2 编程部分：从零实现 MLP
在此部分，我们将使用 PyTorch 实现一个简单的 MLP。

In [1]:
import torch
from torch import nn
from d2l import torch as d2l

# 1. 初始化参数
num_inputs, num_outputs, num_hiddens = 784, 10, 256

W1 = nn.Parameter(torch.randn(num_inputs, num_hiddens, requires_grad=True) * 0.01)
b1 = nn.Parameter(torch.zeros(num_hiddens, requires_grad=True))
W2 = nn.Parameter(torch.randn(num_hiddens, num_outputs, requires_grad=True) * 0.01)
b2 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))

params = [W1, b1, W2, b2]

# 2. 实现 ReLU
def relu(X):
    a = torch.zeros_like(X)
    return torch.max(X, a)

# 3. 实现模型
def net(X):
    X = X.reshape((-1, num_inputs))
    H = relu(X @ W1 + b1)
    return (H @ W2 + b2)

loss = nn.CrossEntropyLoss(reduction='none')

## 3. 模型选择与正则化

### 3.1 理论部分

#### 1. 训练误差与泛化误差
- **训练误差 (Training Error)**：模型在训练数据集上计算得到的误差。
- **泛化误差 (Generalization Error)**：模型应用在同样从原始样本分布中抽取的无限多新样本上时误差的期望。

#### 2. K 折交叉验证 (K-fold Cross-Validation)
将原始训练数据分成 K 个不重叠的子集。然后进行 K 次训练和验证，每次使用一个子集作为验证集，其余 K-1 个子集作为训练集。最后对这 K 次实验的误差取平均。

### 3.2 编程部分：Dropout 实现

In [2]:
def dropout_layer(X, dropout):
    assert 0 <= dropout <= 1
    if dropout == 1:
        return torch.zeros_like(X)
    if dropout == 0:
        return X
    mask = (torch.rand(X.shape) > dropout).float()
    return mask * X / (1.0 - dropout)

# 测试 Dropout
X = torch.arange(16).reshape((2, 8)).float()
print(dropout_layer(X, 0.5))

tensor([[ 0.,  2.,  0.,  0.,  8., 10.,  0., 14.],
        [ 0., 18.,  0., 22., 24.,  0., 28.,  0.]])


## 4. 梯度消失与梯度爆炸

### 4.1 理论部分

#### 1. 原因分析
梯度消失和爆炸是由于深度网络中反向传播时的连乘效应导致的。如果每一层的梯度都小于 1，连乘后梯度会趋近于 0（消失）；如果大于 1，则会趋近于无穷大（爆炸）。

#### 2. ReLU 的作用
ReLU 在正区间的导数为 1，不会像 Sigmoid 在饱和区那样导数趋近于 0，因此能有效缓解梯度消失问题。

## 5. 分布偏移 (Distribution Shift)

### 5.1 理论部分
- **协变量偏移 (Covariate Shift)**：输入 $P(x)$ 改变，但条件概率 $P(y|x)$ 不变。
- **标签偏移 (Label Shift)**：标签 $P(y)$ 改变，但 $P(x|y)$ 不变。

### 5.2 编程部分：重要性采样

In [4]:
import sys
!{sys.executable} -m pip install scikit-learn scipy matplotlib -i https://pypi.tuna.tsinghua.edu.cn/simple --trusted-host pypi.tuna.tsinghua.edu.cn


Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
     --------------------- ------------------ 4.7/8.9 MB 22.0 MB/s eta 0:00:01
     -------------------------------------- - 8.7/8.9 MB 21.5 MB/s eta 0:00:01
     ---------------------------------------- 8.9/8.9 MB 15.8 MB/s  0:00:00

   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- ------------- 2/3 [scikit-learn]
   ------

In [5]:
import numpy as np
import matplotlib.pyplot as plt

# 1. 生成数据
n_train, n_test = 1000, 500
x_train = np.random.normal(-1, 1, n_train)
y_train = 2 * x_train + np.random.normal(0, 1, n_train)

x_test = np.random.normal(2, 1, n_test)
y_test = 2 * x_test + np.random.normal(0, 1, n_test)

# 2. 线性回归（无权重）
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train.reshape(-1, 1), y_train)
y_pred = model.predict(x_test.reshape(-1, 1))
mse_no_weight = np.mean((y_test - y_pred)**2)
print(f"MSE without weights: {mse_no_weight}")

# 3. 重要性采样权重
# w(x) = P_test(x) / P_train(x)
from scipy.stats import norm
w = norm.pdf(x_train, 2, 1) / norm.pdf(x_train, -1, 1)

# 4. 加权线性回归
model_weighted = LinearRegression()
model_weighted.fit(x_train.reshape(-1, 1), y_train, sample_weight=w)
y_pred_weighted = model_weighted.predict(x_test.reshape(-1, 1))
mse_weighted = np.mean((y_test - y_pred_weighted)**2)
print(f"MSE with weights: {mse_weighted}")

MSE without weights: 0.9374197668419563
MSE with weights: 3.009190215983666
